# Streamlit App for Clustering

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# ===============================
# LOAD FILES
# ===============================
df = pd.read_excel("P659_World_development_dataset.xlsx")

model = joblib.load("kmeans_model.joblib")
scaler = joblib.load("scaler.joblib")
pca = joblib.load("pca.joblib")

# ===============================
# UI CONFIG
# ===============================
st.set_page_config(layout="wide")

st.title("🌍 Global Dev Clustering")
st.subheader("Unsupervised ML Project")

# ===============================
# SIDEBAR
# ===============================
st.sidebar.header("Upload Dataset")
uploaded_file = st.sidebar.file_uploader("Upload Excel", type=["xlsx"])

if uploaded_file:
    df = pd.read_excel(uploaded_file)

# ===============================
# COUNTRY SELECT
# ===============================
country = st.selectbox("Select country to inspect", df['Country'])

row = df[df['Country'] == country]

# ===============================
# TRANSFORM + PREDICT
# ===============================
features = df.drop(columns=['Country'], errors='ignore')

scaled = scaler.transform(features)
pca_data = pca.transform(scaled)

clusters = model.predict(pca_data)
df['Cluster'] = clusters

cluster_id = df[df['Country'] == country]['Cluster'].values[0]

# ===============================
# DISPLAY METRICS
# ===============================
st.markdown(f"### 🌐 {country}")
st.write(f"Cluster assigned: **{cluster_id}**")

col1, col2, col3 = st.columns(3)

for i, col in enumerate(features.columns[:9]):
    with [col1, col2, col3][i % 3]:
        st.metric(col, round(row[col].values[0], 2))

# ===============================
# COMPARISON GRAPH
# ===============================
cluster_mean = df[df['Cluster'] == cluster_id].mean(numeric_only=True)

comparison = pd.DataFrame({
    "Country": row.iloc[0][features.columns],
    "Cluster Mean": cluster_mean[features.columns]
})

st.subheader("📊 Country vs Cluster Mean")
st.bar_chart(comparison)

# ===============================
# CLUSTER DISTRIBUTION
# ===============================
st.subheader("Cluster Distribution")
st.bar_chart(df['Cluster'].value_counts())